# DeepLabV3+ Experiment - Segmentasi Biner Daun (untuk Ensemble dengan U-Net)

Notebook ini menjalankan seluruh eksperimen DeepLabV3+ menggunakan **kode dari repo ini**,
bisa dijalankan di **Colab maupun lokal (VS Code / Jupyter)**.

Tujuan akhir: menghasilkan model DeepLabV3+ untuk **segmentasi biner daun**
(`0` = background, `1` = leaf) yang akan di-ensembel dengan model U-Net yang sudah dilatih.
Rancangan mengikuti `docs/ensembel.md`.

## Alur notebook
1. Setup environment (Colab / lokal) + clone repo
2. Install dependensi
3. Download & validasi dataset (`data/imgs`, `data/masks`)
4. Buat / verifikasi split (`splits/{train,val,test}.txt`) - **harus sama dengan U-Net**
5. Konfigurasi training (2 kelas, 256x256)
6. Sanity check DataLoader
7. Training + simpan checkpoint terbaik (berdasarkan Dice foreground di validation set)
8. Evaluasi di **test set** (IoU/Dice/Precision/Recall)
9. **Simpan model & probabilitas untuk ensemble** (bisa disalin ke Google Drive)
10. Ensemble dengan U-Net via `ensemble/fuse_predictions.py`

## 1. Setup Environment & Clone Repo

Mendeteksi lingkungan lalu memakai kode dari repo ini:
- **Colab**: clone ulang repo dari GitHub (tarik kode terbaru, termasuk hasil commit Anda).
- **Lokal**: memakai repo yang sedang terbuka (folder berisi `models/`).

> Pastikan perubahan terbaru (config, data_generator, trainer, predictor, ensemble, splits)
> sudah **di-commit dan di-push** ke GitHub sebelum menjalankan di Colab, karena Colab mengambil
> kode dari repo, bukan dari komputer Anda.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

# ====== KONFIGURASI REPO ======
REPO_URL = "https://github.com/adinmusababa/segmentasi.git"  # ubah jika repo pindah
REPO_BRANCH = "setup"                                         # branch berisi kode terbaru
REPO_DIR_NAME = "deeplabV3-PyTorch"
# ==============================

IN_COLAB = 'google.colab' in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    REPO_DIR = Path('/content') / REPO_DIR_NAME
    if REPO_DIR.exists():
        print("Repo sudah ada di runtime, clone ulang untuk mengambil kode terbaru...")
        shutil.rmtree(REPO_DIR)
    print(f"Cloning {REPO_URL} (branch {REPO_BRANCH}) ...")
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    REPO_DIR = Path.cwd()
    while not (REPO_DIR / "models").exists() and REPO_DIR != REPO_DIR.parent:
        REPO_DIR = REPO_DIR.parent
    if not (REPO_DIR / "models").exists():
        raise RuntimeError("Repo root tidak ditemukan. Jalankan notebook dari dalam folder repo (berisi models/).")
    print(f"Menggunakan repo lokal: {REPO_DIR}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Bekerja di: {os.getcwd()}")
print("models/ ada:", (REPO_DIR / "models").exists())
print("configs/ ada:", (REPO_DIR / "configs").exists())
print("ensemble/ ada:", (REPO_DIR / "ensemble").exists())

## (Opsional) Mount Google Drive

Gunakan jika ingin menyimpan checkpoint dan hasil eksperimen ke Drive agar tetap ada
setelah runtime Colab di-reset. Lewati jika lokal.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    print("Drive ter-mount di:", DRIVE_ROOT)
else:
    DRIVE_ROOT = None
    print("Mode lokal: hasil disimpan di folder repo (lihat bagian 9).")

## 2. Install Dependensi

`%pip` menginstall ke environment kernel yang aktif. Aman dijalankan ulang.
Torch/Torchvision memakai bawaan environment (Colab atau venv lokal) agar versi GPU cocok.

In [ ]:
%pip install -q kagglehub pyyaml tensorboardX tqdm scikit-learn matplotlib pillow numpy

import torch
print("PyTorch:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Download & Validasi Dataset

Dataset **tidak tersimpan di git** (gitignored), jadi harus di-download.
`data/download_dataset.py` memakai `kagglehub` dan aman diulang (melewati jika data sudah ada).

- `data/imgs/` : gambar RGB
- `data/masks/`: mask instance daun (nilai 0..15 = identitas daun, bukan kelas semantic)

In [ ]:
import subprocess, sys
from pathlib import Path

try:
    res = subprocess.run([sys.executable, "data/download_dataset.py"], capture_output=True, text=True, timeout=1800)
    print(res.stdout[-3000:])
    if res.returncode != 0:
        print("STDERR:", res.stderr[-2000:])
except Exception as e:
    print("Download gagal:", e)
    print("Tempatkan gambar/mask secara manual ke data/imgs/ dan data/masks/ dengan nama yang sama.")

imgs = sorted(Path("data/imgs").glob("*.png"))
masks = sorted(Path("data/masks").glob("*.png"))
print(f"Gambar: {len(imgs)} | Mask: {len(masks)}")
assert imgs and masks, "Dataset kosong. Periksa koneksi internet atau pasang data manual."

## 4. Split Train / Val / Test (harus SAMA dengan U-Net)

Split dibaca dari manifest `splits/{train,val,test}.txt` (satu stem per baris, tanpa ekstensi).
Agar ensemble adil, **U-Net dan DeepLabV3 wajib memakai daftar gambar yang identik**.

- Jika file split sudah ada di repo (sudah Anda commit), dipakai apa adanya.
- Jika belum ada, dibuat sekali dengan seed 42 (70/15/15).

> Jika model U-Net sudah punya split sendiri, ganti file di `splits/` dengan split tersebut
> (jangan dibuat ulang) supaya kedua model dievaluasi pada gambar yang sama.

In [ ]:
from pathlib import Path
import subprocess, sys

split_dir = Path("splits")
if all((split_dir / f"{n}.txt").exists() for n in ("train", "val", "test")):
    print("Split manifest sudah ada, dipakai apa adanya.")
else:
    print("Split manifest belum ada. Membuat sekali dengan seed 42 (70/15/15)...")
    subprocess.run([sys.executable, "data_generators/data_generator.py", "--make-splits"], check=True)

for n in ("train", "val", "test"):
    lines = [l for l in (split_dir / f"{n}.txt").read_text().splitlines() if l.strip()]
    print(f"  {n}: {len(lines)}")

imgs = {p.stem for p in Path("data/imgs").glob("*.png")}
masks = {p.stem for p in Path("data/masks").glob("*.png")}
print("Pasangan gambar-mask:", len(imgs & masks), "| tanpa pasangan:", len(imgs ^ masks))
assert imgs == masks, "Ada file tanpa pasangan. Perbaiki dataset dulu."

## 5. Konfigurasi Training (Binary Leaf)

Semua parameter eksperimen dikumpulkan di satu tempat. Nilai default mengikuti
`docs/ensembel.md`: 2 kelas, 256x256, `use_balanced_weights`, `freeze_bn`, seed 42.

> `loss_type='ce_dice'` = Weighted Cross Entropy + Dice Loss (rekomendasi untuk data tidak seimbang).

In [ ]:
import os, torch, yaml
from pathlib import Path

with open("configs/config.yml") as f:
    config = yaml.safe_load(f)

# ====== PARAMETER EKSPERIMEN: sesuaikan di sini ======
EXPERIMENT_NAME = "binary-256-resnet-ce_dice"
NUM_CLASSES      = 2              # wajib 2 (background + leaf) untuk ensemble
BACKBONE         = "resnet"       # resnet / xception / drn / mobilenet
BASE_CROP_SIZE   = 256            # harus sama dengan U-Net
EPOCHS           = 50
BATCH_SIZE       = 4 if torch.cuda.is_available() else 2
LOSS_TYPE        = "ce_dice"      # ce / focal / ce_dice
USE_BALANCED_WEIGHTS = True       # background jauh lebih banyak dari daun
LR               = 0.0005
SEED             = 42             # seed split & training
FREEZE_BN        = True           # batch kecil -> BatchNorm dibekukan
WORKERS          = 0 if os.name == "nt" else 4  # Windows -> 0 agar DataLoader tidak hang
# ====================================================

config["experiment_name"] = EXPERIMENT_NAME
config["dataset"]["base_path"] = str(Path.cwd() / "data")
config["dataset"]["dataset_name"] = "plant_phenotyping"
config["network"]["num_classes"] = NUM_CLASSES
config["network"]["backbone"] = BACKBONE
config["network"]["sync_bn"] = False
config["network"]["freeze_bn"] = FREEZE_BN
config["network"]["use_cuda"] = torch.cuda.is_available()
config["image"]["out_stride"] = 16
config["image"]["base_size"] = BASE_CROP_SIZE
config["image"]["crop_size"] = BASE_CROP_SIZE
config["training"]["workers"] = WORKERS
config["training"]["batch_size"] = BATCH_SIZE
config["training"]["epochs"] = EPOCHS
config["training"]["start_epoch"] = 0
config["training"]["lr"] = LR
config["training"]["lr_scheduler"] = "poly"
config["training"]["momentum"] = 0.9
config["training"]["weight_decay"] = 0.0005
config["training"]["nesterov"] = False
config["training"]["loss_type"] = LOSS_TYPE
config["training"]["use_balanced_weights"] = USE_BALANCED_WEIGHTS
config["training"]["no_val"] = False
config["training"]["val_interval"] = 1
config["training"]["train_on_subset"]["enabled"] = False
config["training"]["train_on_subset"]["dataset_fraction"] = 1.0
config["training"]["weights_initialization"]["use_pretrained_weights"] = False
config["training"]["weights_initialization"]["restore_from"] = ""
config["training"]["tensorboard"]["enabled"] = True
config["training"]["tensorboard"]["log_dir"] = "./tensorboard/"
config["seed"] = SEED

cfg_path = Path("configs") / f"config_{EXPERIMENT_NAME}.yml"
with open(cfg_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config disimpan:", cfg_path)
print(f"  num_classes={NUM_CLASSES}, crop={BASE_CROP_SIZE}, batch={BATCH_SIZE}")
print(f"  epochs={EPOCHS}, loss={LOSS_TYPE}, balanced_weights={USE_BALANCED_WEIGHTS}")
print(f"  backbone={BACKBONE}, freeze_bn={FREEZE_BN}, seed={SEED}")

## 6. Sanity Check DataLoader

Pastikan bentuk tensor sesuai rancangan sebelum training:

```
image = (B, 3, 256, 256)     # RGB, ternormalisasi
label = (B, 256, 256)        # long, hanya berisi {0, 1}
```

In [ ]:
from data_generators.data_generator import initialize_data_loader

train_loader, val_loader, test_loader, nclass = initialize_data_loader(config)
print("nclass:", nclass, "| train/val/test batches:", len(train_loader), len(val_loader), len(test_loader))

batch = next(iter(train_loader))
img, lbl = batch["image"], batch["label"]
print("image:", tuple(img.shape), "label:", tuple(lbl.shape), "dtype:", lbl.dtype)
print("label unik:", torch.unique(lbl).tolist(), "(harus [0, 1])")

assert nclass == NUM_CLASSES
assert tuple(img.shape[1:]) == (3, BASE_CROP_SIZE, BASE_CROP_SIZE)
assert lbl.min() >= 0 and lbl.max() <= 1
print("OK: DataLoader siap.")

## 7. Training

Menjalankan seluruh epoch training + validasi.
Checkpoint terbaik disimpan ke `experiments/checkpoint_best.pth.tar` berdasarkan
**Dice foreground (leaf) di validation set**, bukan test set.

```text
eksperimen  minimum:
1. DeepLabV3 saja        -> (notebook ini)
2. U-Net saja            -> repo U-Net
3. Ensemble alpha=0.5 th=0.5
4. Ensemble alpha & threshold terbaik dari validation set
```

In [ ]:
from pathlib import Path
from trainers.trainer import Trainer

Path("experiments").mkdir(parents=True, exist_ok=True)
config["checkname"] = "deeplab-" + str(config["network"]["backbone"])

trainer = Trainer(config)
print(f"Mulai epoch {trainer.config['training']['start_epoch']} s/d {trainer.config['training']['epochs'] - 1}")

for epoch in range(trainer.config["training"]["start_epoch"], trainer.config["training"]["epochs"]):
    trainer.training(epoch)
    if not trainer.config["training"]["no_val"] and epoch % config["training"]["val_interval"] == (config["training"]["val_interval"] - 1):
        trainer.validation(epoch)

trainer.writer.close()
print("Training selesai.")
best_ckpt = Path("experiments/checkpoint_best.pth.tar")
last_ckpt = Path("experiments/checkpoint_last.pth.tar")
print("Checkpoint terbaik:", best_ckpt, "| ada:", best_ckpt.exists())

## 8. Evaluasi di Test Set

Mengevaluasi checkpoint terbaik pada **test set** (bukan val set).
Melaporkan metrik foreground (leaf): IoU, Dice, Precision, Recall + confusion matrix.

In [ ]:
from predictors.predictor import Predictor

best_ckpt = Path("experiments/checkpoint_best.pth.tar")
ckpt = best_ckpt if best_ckpt.exists() else last_ckpt
print("Checkpoint untuk evaluasi:", ckpt)

predictor = Predictor(config, checkpoint_path=str(ckpt))
metrics = predictor.inference_on_test_set()

for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 9. Simpan Model & Hasil untuk Ensemble

Mengekspor artefak ke `ensemble_ready/<experiment>/`:

- `checkpoint_best.pth.tar` : checkpoint model
- `config.yml`              : konfigurasi yang dipakai
- `test_metrics.json`       : metrik test set
- `test_probs/*.npy`        : probabilitas foreground per gambar test (input ensemble)
- `test_masks/*.png`        : mask biner per gambar test

Di Colab, folder hasil **disalin ke Google Drive** agar tidak hilang saat runtime di-reset.

In [ ]:
import json, shutil
import numpy as np
from pathlib import Path
from PIL import Image
from data_generators.data_generator import load_split_names

out = Path("ensemble_ready") / EXPERIMENT_NAME
out.mkdir(parents=True, exist_ok=True)

# 1) metrik test set
with open(out / "test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)

# 2) config yang dipakai
shutil.copy(Path("configs") / f"config_{EXPERIMENT_NAME}.yml", out / "config.yml")

# 3) checkpoint terbaik
ckpt_dest = out / Path(ckpt).name
shutil.copy(ckpt, ckpt_dest)
print("Checkpoint:", ckpt_dest)

# 4) probabilitas foreground + mask biner untuk test set
test_stems = load_split_names(Path("splits"), "test")
prob_dir, mask_dir = out / "test_probs", out / "test_masks"
prob_dir.mkdir(exist_ok=True); mask_dir.mkdir(exist_ok=True)

for stem in test_stems:
    img_path = Path("data/imgs") / f"{stem}.png"
    prob = predictor.predict_probability(str(img_path))           # (H,W) float32 [0,1]
    np.save(prob_dir / f"{stem}.npy", prob)
    mask = (prob >= 0.5).astype(np.uint8)                          # mask biner
    Image.fromarray(mask).save(mask_dir / f"{stem}.png")

print(f"Probabilitas ({len(test_stems)} file) -> {prob_dir}")
print(f"Mask biner -> {mask_dir}")

# 5) ringkasan hasil
summary = {
    "experiment": EXPERIMENT_NAME,
    "seed": SEED,
    "split": {n: len(load_split_names(Path("splits"), n)) for n in ("train", "val", "test")},
    "model": {"backbone": BACKBONE, "num_classes": NUM_CLASSES, "crop": BASE_CROP_SIZE},
    "training": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "loss": LOSS_TYPE, "lr": LR},
    "test_metrics": metrics,
}
with open(out / "summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

# 6) salin ke Google Drive jika aktif
if IN_COLAB and DRIVE_ROOT is not None:
    try:
        drive_dest = DRIVE_ROOT / "segmentation_experiments" / EXPERIMENT_NAME
        drive_dest.mkdir(parents=True, exist_ok=True)
        shutil.copytree(out, drive_dest, dirs_exist_ok=True)
        print("Disalin ke Google Drive:", drive_dest)
    except Exception as e:
        print("Lewati simpan ke Drive:", e)

print("Semua artefak siap di:", out.resolve())

## 10. Visualisasi Prediksi

Menampilkan gambar asli, ground-truth biner, probabilitas daun, dan mask prediksi
untuk beberapa gambar test.

Periksa minimal:
- gambar asli
- ground-truth biner
- prediksi DeepLabV3
- area false positive / false negative

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

def binarize_gt(p):
    m = np.asarray(Image.open(p))
    if m.ndim == 3:
        m = np.any(m != 0, axis=2)
    return (m != 0).astype(np.uint8)

show = test_stems[:6]
n = len(show)
fig, axes = plt.subplots(n, 5, figsize=(18, 3 * n))
if n == 1:
    axes = axes[None, :]

for r, stem in enumerate(show):
    img = np.asarray(Image.open(Path("data/imgs") / f"{stem}.png").convert("RGB"))
    gt = binarize_gt(Path("data/masks") / f"{stem}.png")
    prob = predictor.predict_probability(str(Path("data/imgs") / f"{stem}.png"))
    pred = (prob >= 0.5).astype(np.uint8)
    fp = (pred == 1) & (gt == 0)
    fn = (pred == 0) & (gt == 1)

    axes[r, 0].imshow(img); axes[r, 0].set_title("Original"); axes[r, 0].axis("off")
    axes[r, 1].imshow(gt, cmap="gray"); axes[r, 1].set_title("GT"); axes[r, 1].axis("off")
    axes[r, 2].imshow(prob, cmap="hot", vmin=0, vmax=1); axes[r, 2].set_title("Prob leaf"); axes[r, 2].axis("off")
    axes[r, 3].imshow(pred, cmap="gray"); axes[r, 3].set_title("Pred"); axes[r, 3].axis("off")
    # error map: hijau TP, kuning FP, biru FN
    err = np.zeros((*gt.shape, 3), dtype=np.uint8)
    err[fp] = [255, 200, 0]   # kuning = false positive
    err[fn] = [0, 150, 255]   # biru   = false negative
    axes[r, 4].imshow(img); axes[r, 4].imshow(err, alpha=0.5); axes[r, 4].set_title("FP/FN"); axes[r, 4].axis("off")

plt.tight_layout()
plt.show()

## 11. Ensemble dengan U-Net

Gabungkan probabilitas foreground U-Net (`unet_prob`) dan DeepLabV3 (`deeplab_prob`):

```python
ensemble_prob = alpha * unet_prob + (1 - alpha) * deeplab_prob
ensemble_mask = (ensemble_prob >= threshold)
```

- `alpha` besar -> lebih percaya U-Net; kecil -> lebih percaya DeepLabV3
- `alpha` dan `threshold` **hanya dicari dari validation set**

Langkah:
1. Isi fungsi `unet_prob_fn` di bawah dengan panggilan ke model U-Net Anda.
2. Jalankan pencarian `alpha`/`threshold` (cell berikutnya).
3. Evaluasi final di test set.</think>

> Contoh U-Net yang menghasilkan 2 channel logit:
> `unet_logits = unet_model(preprocess(image)); prob = softmax(unet_logits, dim=1)[0, 1]`

In [ ]:
# ====== ISI fungsi ini dengan model U-Net Anda ======
import numpy as np

def unet_prob_fn(image_path):
    # Kembalikan probabilitas daun (H, W) float32 di resolusi asli dari U-Net.
    # Contoh untuk model 2-channel softmax (sesuai docs/ensembel.md):
    #   import torch, torch.nn.functional as F
    #   from PIL import Image
    #   img = Image.open(image_path).convert("RGB")
    #   x = preprocess_unet(img).unsqueeze(0)                 # (1,3,256,256) preprocessing U-Net
    #   with torch.no_grad():
    #       logits = unet_model(x)                            # (1,2,256,256)
    #   prob = F.softmax(logits, dim=1)[0, 1].cpu().numpy()  # (256,256)
    #   prob = np.array(Image.fromarray(prob).resize(Image.open(image_path).size, Image.BILINEAR), dtype=np.float32)
    #   return prob
    raise NotImplementedError("Isi unet_prob_fn dengan prediksi U-Net Anda, lalu jalankan ulang.")

# verifikasi apakah fungsi sudah diisi
_try = None
try:
    _try = unet_prob_fn(str(Path("data/imgs") / test_stems[0]))
    HAVE_UNET = isinstance(_try, np.ndarray)
except NotImplementedError:
    HAVE_UNET = False

print("U-Net tersedia:", HAVE_UNET)
if not HAVE_UNET:
    print("SKIP: isi unet_prob_fn di atas terlebih dahulu sebelum menjalankan cell ensemble.")

### 11a. Cari alpha & threshold terbaik (validation set)

In [ ]:
if HAVE_UNET:
    from ensemble.fuse_predictions import EnsembleFusion, search_alpha_threshold

    deep_ckpt = str(out / Path(ckpt).name)

    # gunakan split val untuk tuning
    best = search_alpha_threshold(
        config, deep_ckpt, unet_prob_fn,
        split_dir=Path("splits"), data_dir=Path("data"),
    )
    print("Hasil pencarian (validation):")
    print(f"  alpha={best['alpha']}, threshold={best['threshold']}, val_dice={best['best_dice']:.4f}")

    # evaluasi ensemble di TEST SET dengan parameter terpilih
    from ensemble.fuse_predictions import evaluate_masks
    fuser = EnsembleFusion(config, deep_ckpt, unet_prob_fn)

    test_stems_t = load_split_names(Path("splits"), "test")
    def ens_mask(stem):
        m, _ = fuser.predict_mask(str(Path("data/imgs") / f"{stem}.png"),
                                  alpha=best["alpha"], threshold=best["threshold"])
        return m

    ens_metrics = evaluate_masks(Path("data/masks"), test_stems_t, ens_mask)
    print("Ensemble (test set):")
    print(f"  Dice={ens_metrics['Dice_leaf']:.4f} IoU={ens_metrics['IoU_leaf']:.4f} "
          f"Prec={ens_metrics['Precision']:.4f} Rec={ens_metrics['Recall']:.4f}")
else:
    print("Lewati (U-Net belum diisi).")

## 12. TensorBoard (Opsional)

Menjalankan TensorBoard untuk melihat kurva loss, Dice, dan IoU per epoch.

In [ ]:
if IN_COLAB:
    %load_ext tensorboard
    %tensorboard --logdir ./tensorboard --port 6006
else:
    print("Jalankan di terminal:  tensorboard --logdir ./tensorboard")

## 13. Ringkasan & Log Eksperimen

Mencetak ringkasan sesuai protokol eksperimen `docs/ensembel.md` untuk dicatat di laporan.

In [ ]:
import json
s = json.loads((out / "summary.json").read_text())
print("=" * 60)
print("PROTOKOL EKSPERIMEN")
print("=" * 60)
print(f"experiment_id        : {s['experiment']}")
print(f"seed                 : {s['seed']}")
print(f"split                : {s['split']}")
print(f"backbone/num_classes : {s['model']['backbone']} / {s['model']['num_classes']}")
print(f"image_size           : {s['model']['crop']} x {s['model']['crop']}")
print(f"epochs/batch/lr      : {s['training']['epochs']} / {s['training']['batch_size']} / {s['training']['lr']}")
print(f"loss                 : {s['training']['loss']}")
print(f"checkpoint           : {out / Path(ckpt).name}")
tm = s["test_metrics"]
print(f"test Dice/IoU/Prec/Rec: {tm['Dice_leaf']:.4f} / {tm['IoU_leaf']:.4f} / "
      f"{tm['Precision']:.4f} / {tm['Recall']:.4f}")
print("=" * 60)
print("Langkah selanjutnya: bandingkan U-Net saja vs DeepLabV3 saja vs ensemble,")
print("dan hanya nyatakan ensemble lebih baik jika metrik test set meningkat.")